In [ ]:
"""
Ellis Mullen - 21334498

1. Removes duplicate images using perceptual hashing.
2. Resizes all images to 128x128 pixels.
3. Assigns labels:
    robin      -> 1
    not_robin  -> 0
4. Creates a stratified 80/20 train-test split.
5. Generates 5 stratified cross-validation folds.
6. Saves all information to a CSV file.
"""

import os
from PIL import Image
import imagehash
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

# All configs defined here
DATASET_PATH = "Big_Dataset"
IMAGE_SIZE = (128, 128)
OUTPUT_CSV = "bird_dataset.csv"
TEST_SIZE = 0.20
N_FOLDS = 5
RANDOM_STATE = 42

# Perceptual hashing for detecting and removing duplicate images
seen_hashes = {}
duplicate_files = []

for root, _, files in os.walk(DATASET_PATH):
    for file in files:

        #Get full image path
        image_path = os.path.join(root, file)
        try:
            image = Image.open(image_path)
            #Create a perceptual hash based on image content
            image_hash = str(imagehash.phash(image))
            #Check if the hash has already been seen
            if image_hash in seen_hashes:
                duplicate_files.append(image_path)
            #If its new, add it to the seen_hashes dictionary
            else:
                seen_hashes[image_hash] = image_path
        #Error handling
        except Exception as error:
            print(f"Could not process {image_path}")
            print(error)
#If any duplicate files were found, remove them
for image_path in duplicate_files:
    os.remove(image_path)

#print the number of duplicate images removed
print(f"{len(duplicate_files)} duplicate images removed.")

print("Resizing All Images to 128x128 pixels")


for root, _, files in os.walk(DATASET_PATH):
    for file in files:

        image_path = os.path.join(root, file)
        try:
            image = Image.open(image_path)
            image = image.convert("RGB")
            image = image.resize(IMAGE_SIZE)
            image.save(image_path)

        except Exception as error:
            print(f"Could not resize {image_path}")
            print(error)

print("Image resizing done.")

print("Creating dataset records...")

dataset_records = []

for root, _, files in os.walk(DATASET_PATH):
    for file in files:

        image_path = os.path.join(root, file)
        folder_name = os.path.basename(root).lower()
        if folder_name == "robin":
            label = 1
        elif folder_name == "not_robin":
            label = 0
        else:
            continue

        dataset_records.append({
            "filepath": image_path,
            "label": label
        })

dataset_dataframe = pd.DataFrame(dataset_records)

print(f"Images found: {len(dataset_dataframe)}")

#Splitting the dataset into training and testing sets
training_dataframe, testing_dataframe = train_test_split(
    dataset_dataframe,
    test_size=TEST_SIZE,
    stratify=dataset_dataframe["label"],
    random_state=RANDOM_STATE
)

#Resetting the index of the dataframes and adding a new column for fold assignment
training_dataframe = training_dataframe.reset_index(drop=True)
training_dataframe["fold"] = -1

cross_validator = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

#Assigning fold numbers to the training set
for fold_number, (_, validation_indices) in enumerate(
    cross_validator.split(
        training_dataframe["filepath"],
        training_dataframe["label"]
    )
):

    training_dataframe.loc[
        validation_indices,
        "fold"
    ] = fold_number

#Assigning a fold number of -1 to the testing set
testing_dataframe = testing_dataframe.copy()
testing_dataframe["fold"] = "test"

#Combine the dataframes and saving them to .csv
final_dataframe = pd.concat(
    [training_dataframe, testing_dataframe],
    ignore_index=True
)

final_dataframe = final_dataframe[
    ["filepath", "label", "fold"]
]

final_dataframe.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Preprocessing complete.")
print(f"Total Images: {len(final_dataframe)}")
print(f"Training Images: {len(training_dataframe)}")
print(f"Testing Images: {len(testing_dataframe)}")

Looking for duplicates...
1 duplicate images removed.
Resizing All Images to 128x128 pixels
Image resizing done.
Creating dataset records...
Images found: 1951

Preprocessing complete.
CSV file saved as: bird_dataset.csv

Dataset Summary
--------------------
Total Images: 1951
Training Images: 1560
Testing Images: 391
